# Personal Finance AI Agent

An AI-powered Personal Finance Assistant built with **LangChain Agents**, custom tools and **chat memory**.

### Tools Implemented
| # | Tool | What it does |
|---|---|---|
| 1 | EMI Calculator | EMI, total interest, total payment, DTI ratio |
| 2 | SIP Calculator | Future value, total invested, wealth gained |
| 3 | Budget Allocator | 50-30-20 split, surplus, savings rate |
| 4 | Goal Planner | Integrated Goal Planner (budget capacity, dynamic returns by tenure, web inflation) |
| 5 | Python REPL Calculator | Runs any Python expression for ad-hoc arithmetic |
| 6 | Financial News Search | Fetches live financial news & market data via DuckDuckGoSearch |

---
## Step 1 — Imports & Setup

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_experimental.tools import PythonREPLTool
from langchain_community.tools import DuckDuckGoSearchRun

load_dotenv()

In [ ]:
# Initializing LLM
#llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

---
## Step 2 — Custom Finance Tool Development

Design Principle: Custom tools are responsible only for performing deterministic computations and retrieving relevant financial information. The AI-Agent’s LLM layer interprets the tool outputs, performs reasoning, and generates meaningful financial insights for the user.

### Tool 1: EMI Calculator

This tool calculates the monthly loan repayment obligation and evaluates repayment affordability by computing the **Equated Monthly Installment (EMI)** and **Debt-to-Income (DTI) ratio**. It helps the AI-Agent analyze loan scenarios such as home loans, car loans, and personal loans.

### Inputs:
- **Loan Amount (`loan_amount`)**
  - Principal amount borrowed from the lender (in rupees).
  - Example: ₹10,00,000

- **Annual Interest Rate (`annual_interest_rate`)**
  - Yearly interest rate charged on the loan, expressed as a percentage.
  - Example: 8%

- **Loan Tenure (`tenure_years`)**
  - Total repayment period of the loan (in years).
  - Example: 20 years

- **Monthly Income (`monthly_income`)**
  - Borrower's monthly income used for calculating the DTI ratio.
  - Optional input.
  - Example: ₹80,000/month

### Outputs:
- **Monthly EMI**
  - Fixed monthly repayment amount required throughout the loan tenure.

- **Total Interest Payable**
  - Total interest amount paid over the complete repayment period.

- **Total Payment**
  - Overall repayment amount including principal and interest.

- **Debt-to-Income (DTI) Ratio**
  - Percentage of monthly income allocated towards loan repayment.

- **DTI Risk Status**
  - Categorizes repayment affordability based on DTI:
    - **Excellent:** DTI ≤ 36%
    - **Manageable:** 36% < DTI ≤ 43%
    - **High Risk:** DTI > 43%

In [ ]:
@tool
def calculate_emi(loan_amount: float, annual_interest_rate: float, tenure_years: float, monthly_income: float = 0.0) -> str:
    """
    Calculate the monthly EMI for a loan and the Debt-to-Income (DTI) ratio.

    Use this tool when the user asks about loan EMI, monthly payment for a loan,
    or how much they need to pay per month for a home/car/personal loan.

    Args:
        loan_amount: Principal loan amount in rupees (e.g. 1000000 for 10 lakh)
        annual_interest_rate: Annual interest rate as percentage (e.g. 8 for 8%)
        tenure_years: Loan tenure in years (e.g. 20 for 20 years)
        monthly_income: Monthly income in rupees for DTI calculation (optional, e.g. 80000)

    Returns:
        Computed EMI, total interest, total payment and DTI ratio as structured text.
    """
    P        = loan_amount
    r_annual = annual_interest_rate
    t_years  = tenure_years
    income   = monthly_income

    # EMI Formula: P x r x (1+r)^n / ((1+r)^n - 1)
    r = (r_annual / 12) / 100
    n = int(t_years * 12)

    emi            = P * r * ((1 + r) ** n) / (((1 + r) ** n) - 1) if r != 0 else P / n
    total_payment  = emi * n
    total_interest = total_payment - P

    # Debt-to-Income Ratio
    dti        = (emi / income * 100) if income > 0 else 0.0
    dti_status = "Excellent" if dti <= 36 else ("Manageable" if dti <= 43 else "High Risk")

    return (
        f"EMI Calculation Results\n"
        f"{'='*40}\n"
        f"Loan Amount     : Rs. {P:,.2f}\n"
        f"Interest Rate   : {r_annual}% per annum\n"
        f"Tenure          : {t_years} years ({n} months)\n"
        f"{'='*40}\n"
        f"Monthly EMI     : Rs. {emi:,.2f}\n"
        f"Total Interest  : Rs. {total_interest:,.2f}\n"
        f"Total Payment   : Rs. {total_payment:,.2f}\n"
        f"{'='*40}\n"
        f"Monthly Income  : Rs. {income:,.2f}\n"
        f"DTI Ratio       : {dti:.2f}%\n"
        f"DTI Status      : {dti_status}\n"
    )

### Tool 2: SIP Investment Calculator
 
This tool calculates the projected growth of a Systematic Investment Plan (SIP) by estimating the future value of monthly investments based on expected returns and investment duration.

### Inputs:
- **Monthly Investment (`monthly_investment`)**  
  - The fixed amount invested every month through SIP (in rupees).

- **Annual Return Rate (`annual_return_rate`)**  
  - The expected yearly return from the investment, expressed as a percentage.

- **Investment Tenure (`tenure_years`)**  
  - The total duration for which the SIP investment is continued (in years).

### Outputs:
- **Total Invested Amount**  
  - Total capital contributed by the investor over the complete investment period.

- **Future Value**  
  - Projected maturity value of the SIP investment after applying expected returns.

- **Wealth Gained**  
  - Additional value generated over the total invested amount.

- **Gain Multiplier**  
  - Ratio indicating how many times the investment has grown compared to the invested capital.

In [ ]:
@tool
def calculate_sip(monthly_investment: float, annual_return_rate: float, tenure_years: float) -> str:
    """
    Calculate the future value of a Systematic Investment Plan (SIP).

    Use this tool when the user asks about SIP returns, monthly investment growth,
    mutual fund projections, or how much money they will accumulate by investing monthly.

    Args:
        monthly_investment: Monthly SIP amount in rupees (e.g. 5000)
        annual_return_rate: Expected annual return as percentage (e.g. 12 for 12%)
        tenure_years: Investment duration in years (e.g. 10)

    Returns:
        Computed total invested, future value and wealth gained as structured text.
    """
    P        = monthly_investment
    r_annual = annual_return_rate
    t_years  = tenure_years

    # Future Value Formula: P x ((1+r)^n - 1) / r
    r = (r_annual / 12) / 100
    n = int(t_years * 12)

    fv              = P * (((1 + r) ** n - 1) / r) if r != 0 else P * n
    total_invested  = P * n
    wealth_gained   = fv - total_invested
    gain_multiplier = fv / total_invested if total_invested > 0 else 1

    return (
        f"SIP Investment Results\n"
        f"{'='*40}\n"
        f"Monthly Investment : Rs. {P:,.2f}\n"
        f"Expected Return    : {r_annual}% per annum\n"
        f"Investment Period  : {t_years} years ({n} months)\n"
        f"{'='*40}\n"
        f"Total Invested     : Rs. {total_invested:,.2f}\n"
        f"Future Value       : Rs. {fv:,.2f}\n"
        f"Wealth Gained      : Rs. {wealth_gained:,.2f}\n"
        f"Gain Multiplier    : {gain_multiplier:.2f}x\n"
    )

### Tool 3: Budget Allocator (50-30-20 Rule)

This tool creates a monthly budget plan using the **50-30-20 rule** and evaluates financial health based on income, expenses, and loan obligations. The AI-Agent uses these results to provide personalized budgeting insights.

### Inputs:
- **Monthly Income:** Total take-home income.
- **Monthly Expenses:** Regular monthly spending.
- **Loan EMI:** Existing loan repayment obligations.

### Outputs:
- **Total Expenses:** Combined expenses and EMI.
- **Monthly Surplus:** Remaining income after expenses.
- **Budget Allocation:** Split into Needs (50%), Wants (30%), and Savings (20%).
- **Emergency Fund & Investment Allocation:** Savings distribution.
- **Savings Rate & Status:** Financial health assessment based on saving capacity.

In [ ]:
@tool
def calculate_budget(monthly_income: float, monthly_expenses: float = 0.0, loan_emi: float = 0.0) -> str:
    """
    Create a monthly budget plan using the 50-30-20 rule and compute savings health.

    Use this tool when the user asks to create a budget, allocate salary,
    plan monthly expenses, or distribute their income.

    The 50-30-20 Rule:
        - 50% Needs  (rent, food, utilities, EMIs)
        - 30% Wants  (entertainment, dining out, shopping)
        - 20% Savings & Investments

    Args:
        monthly_income: Total monthly take-home income in rupees (e.g. 80000)
        monthly_expenses: Current monthly expenses in rupees (optional, e.g. 40000)
        loan_emi: Monthly loan EMI in rupees (optional, e.g. 8000)

    Returns:
        Computed budget split, surplus, savings rate and savings status as structured text.
    """
    income   = monthly_income
    expenses = monthly_expenses
    emi      = loan_emi

    total_expenses = expenses + emi
    surplus        = income - total_expenses
    savings_rate   = (surplus / income) if income > 0 else 0

    # 50-30-20 Rule
    needs   = income * 0.50
    wants   = income * 0.30
    savings = income * 0.20

    # Adjust if surplus is too low to sustain 20% savings
    if surplus < savings:
        savings = max(0, surplus)
        wants   = max(0, income - (needs + savings))

    emergency_fund = savings * 0.40
    investments    = savings * 0.60

    if savings_rate >= 0.2:
        status = "Strong"
    elif savings_rate >= 0.1:
        status = "Moderate"
    else:
        status = "Weak"

    return (
        f"Budget Allocation Results (50-30-20 Rule)\n"
        f"{'='*45}\n"
        f"Monthly Income    : Rs. {income:,.2f}\n"
        f"Total Expenses    : Rs. {total_expenses:,.2f}\n"
        f"Monthly Surplus   : Rs. {surplus:,.2f}\n"
        f"{'='*45}\n"
        f"Needs (50%)       : Rs. {needs:,.2f}\n"
        f"Wants (30%)       : Rs. {wants:,.2f}\n"
        f"Savings (20%)     : Rs. {savings:,.2f}\n"
        f"  Emergency Fund  : Rs. {emergency_fund:,.2f}\n"
        f"  Investments     : Rs. {investments:,.2f}\n"
        f"{'='*45}\n"
        f"Savings Rate      : {savings_rate*100:.1f}%\n"
        f"Savings Status    : {status}\n"
    )

### Tool 4: Integrated Goal Planner

This tool calculates the required monthly SIP investment for achieving a financial goal by considering the target amount, investment duration, inflation, expected returns, and user's savings capacity. It also evaluates goal feasibility and suggests an investment allocation strategy.

### Inputs:
- **Target Amount:** Desired future goal value.
- **Goal Duration:** Time available to achieve the goal (years).
- **Monthly Income:** Income used to calculate savings capacity.
- **Monthly Expenses:** Current monthly spending.
- **Loan EMI:** Existing loan obligations.
- **Inflation Rate:** Rate used for future value adjustment.
- **Risk Appetite:** Investor risk preference (Low/Moderate/High).
- **Expected Return:** Optional annual return assumption.

### Outputs:
- **Inflation-Adjusted Target:** Future cost of the goal.
- **Required Monthly SIP:** Investment needed to achieve the goal.
- **Budget Surplus:** Available monthly investment capacity.
- **Goal Feasibility:** Whether the goal is achievable.
- **Recommended Asset Allocation:** Suggested Equity, Debt, and Gold distribution.

In [ ]:
@tool
def calculate_goal_savings(
    target_amount: float,
    years_to_goal: float,
    monthly_income: float = 0.0,
    monthly_expenses: float = 0.0,
    loan_emi: float = 0.0,
    fetched_inflation_rate: float = 6.0,
    risk_appetite: str = "Moderate",
    expected_annual_return: float = 0.0
) -> str:
    """
    Calculate required monthly SIP for a financial goal by integrating:
    1. Monthly savings capacity computed via budget surplus (income - expenses - emi).
    2. Dynamic expected annual return computed automatically based on goal tenure.
    3. Inflation rate fetched live from web search.

    Use this tool when the user asks to plan a financial goal (e.g. house, retirement, education, car).

    Arguments:
        target_amount: Goal target amount in rupees (e.g. 2000000 for 20 lakh)
        years_to_goal: Time horizon in years (e.g. 10)
        monthly_income: Income for budget surplus calculation (optional, e.g. 80000)
        monthly_expenses: Expenses for budget surplus calculation (optional, e.g. 35000)
        loan_emi: Active loan EMI in rupees (optional, e.g. 8000)
        fetched_inflation_rate: Inflation rate fetched from web search (e.g. 5.5 for 5.5%)
        risk_appetite: 'Low', 'Moderate', or 'High' (optional)
        expected_annual_return: Override annual return rate (optional, 0.0 to auto-compute from tenure)

    Returns:
        Required monthly SIP, budget surplus capacity, dynamic return rate, inflation target & feasibility.
    """
    # 1. Compute Monthly Savings Capacity from Budget
    total_expenses = monthly_expenses + loan_emi
    monthly_surplus = max(0.0, monthly_income - total_expenses) if monthly_income > 0 else 0.0

    # 2. Dynamic Expected Annual Return based on Tenure & Risk Appetite
    if expected_annual_return > 0:
        exp_return = expected_annual_return
    elif years_to_goal < 3:
        exp_return = 7.0 if risk_appetite == "Low" else 8.0
    elif years_to_goal <= 7:
        exp_return = 9.0 if risk_appetite == "Low" else (10.5 if risk_appetite == "Moderate" else 12.0)
    else:
        exp_return = 10.0 if risk_appetite == "Low" else (12.0 if risk_appetite == "Moderate" else 14.0)

    # 3. Inflation Adjustment & Reverse SIP Calculation
    inflation = fetched_inflation_rate
    inflated_target = target_amount * ((1 + inflation / 100) ** years_to_goal)

    r = exp_return / 1200   # monthly rate
    n = int(years_to_goal * 12)

    if r > 0:
        required_monthly = inflated_target * r / (((1 + r) ** n - 1) * (1 + r))
    else:
        required_monthly = inflated_target / n if n else 0.0

    # 4. Feasibility Evaluation against Budget Surplus
    if monthly_surplus > 0:
        if monthly_surplus >= required_monthly:
            feasibility = "Achievable"
            shortfall = 0.0
            status_note = f"Your budget surplus (Rs. {monthly_surplus:,.2f}) fully covers the required SIP."
        else:
            feasibility = "Not Feasible"
            shortfall = required_monthly - monthly_surplus
            status_note = f"Monthly shortfall of Rs. {shortfall:,.2f}. Budget surplus (Rs. {monthly_surplus:,.2f}) is below required SIP."
    else:
        feasibility = "N/A — No budget inputs provided"
        shortfall = 0.0
        status_note = "Provide monthly income & expenses to assess budget feasibility."

    # Asset Allocation Recommendation
    allocation = {
        "Equity": "70%" if years_to_goal >= 7 else ("40%" if years_to_goal >= 3 else "10%"),
        "Debt":   "20%" if years_to_goal >= 7 else ("50%" if years_to_goal >= 3 else "80%"),
        "Gold":   "10%" if years_to_goal >= 7 else ("10%" if years_to_goal >= 3 else "10%")
    }

    return (
        f"Integrated Goal & Budget Planner Results\n"
        f"{'='*50}\n"
        f"Target Amount (Today)  : Rs. {target_amount:,.2f}\n"
        f"Time Horizon           : {years_to_goal} years ({n} months)\n"
        f"Web Inflation Rate     : {inflation}% per annum\n"
        f"Inflation-Adj. Target  : Rs. {inflated_target:,.2f}\n"
        f"Dynamic Expected Return: {exp_return}% p.a. (based on {years_to_goal}y horizon)\n"
        f"{'='*50}\n"
        f"Required Monthly SIP   : Rs. {required_monthly:,.2f}\n"
        f"Monthly Budget Surplus : Rs. {monthly_surplus:,.2f}\n"
        f"Goal Feasibility       : {feasibility}\n"
        f"Status Note            : {status_note}\n"
        f"{'='*50}\n"
        f"Recommended Asset Mix  : Equity {allocation['Equity']} | Debt {allocation['Debt']} | Gold {allocation['Gold']}\n"
    )

### Tool 5: Python REPL Calculator

This tool provides a Python execution environment for performing dynamic arithmetic and financial calculations required by the AI-Agent.

### Inputs: 
- **Python Code / Expressions:** Mathematical or computational instructions.

### Outputs:
- **Calculation Results:** Generated numerical outputs.

In [ ]:
# PythonREPLTool 
python_repl_tool = PythonREPLTool()

print("Python REPL Calculator ready!")

### Tool 6: Financial News Search

This tool enables the AI-Agent to retrieve **real-time financial information, market updates, and external data** through web search.

### Inputs:
- **Search Query:** User-specific financial information request.

### Outputs:
- **Search Results:** Relevant financial news, data, and information from web.

In [ ]:
# DuckDuckGoSearchRun 
search_tool = DuckDuckGoSearchRun()

print("Financial News Search ready!")

---
## Step 3 — Building the Finance Agent

This section configures the **Personal Finance AI-Agent** by registering specialized finance tools and defining its reasoning workflow. The agent uses tool-driven calculations for EMI analysis, SIP projections, budgeting, goal planning, arithmetic operations, and live financial information retrieval.

### Registered Tools:
- **EMI Calculator:** Loan repayment and DTI analysis.
- **SIP Calculator:** Investment growth and wealth projection.
- **Budget Allocator:** 50-30-20 budget planning and savings analysis.
- **Goal Planner:** Goal-based SIP planning with inflation and feasibility analysis.
- **Python REPL:** Dynamic arithmetic calculations.
- **Web Search:** Real-time financial data and market updates.

### Agent Workflow:
The AI-Agent selects the appropriate tool based on the user query, uses computed outputs for reasoning, and generates personalized financial insights while avoiding manual estimations.

**Design Principle:**  
Tools handle calculations and data retrieval, while the LLM acts as the reasoning layer responsible for analysis, interpretation, and user-facing recommendations.

In [ ]:
# Register all finance tools
finance_tools = [
    calculate_emi,
    calculate_sip,
    calculate_budget,
    calculate_goal_savings,
    python_repl_tool,   # PythonREPLTool for ad-hoc arithmetic
    search_tool,        # DuckDuckGoSearchRun for live financial news
]

# Create the Finance Agent
finance_agent = create_agent(
    model=llm,
    tools=finance_tools,
    system_prompt="""
    You are a Personal Finance AI Assistant designed specifically for Indian users.

   Your primary capabilities include:
    - Loan EMI calculations and DTI analysis
    - SIP investment projections and wealth planning
    - Monthly budget planning (50-30-20 rule)
    - Integrated Financial Goal Planning (with budget surplus, dynamic tenure returns, and live inflation rate)
    - General arithmetic and percentage calculations
    - Live financial news, stock market updates, RBI rates,
      mutual fund data and inflation trends from the web

    Operational Workflow:
    1. Invoke the appropriate tool to generate accurate numerical outputs.
       All calculations must be tool-driven, manual or estimated calculations are not permitted.
       - For Loan EMI       : use calculate_emi
       - For SIP returns    : use calculate_sip
       - For Budget split   : use calculate_budget
       - For arithmetic     : use Python_REPL
       - For news/rates     : use duckduckgo_search
       - For Goal Planning  : first search web for current inflation rate using 'duckduckgo_search' if not provided,
                              then call 'calculate_goal_savings' passing income, expenses, EMI, and fetched inflation rate.

    2. Based on the tool result, provide a complete response:
       - For EMI    : evaluate the DTI ratio and assess the sustainability of the user’s debt obligations
       - For SIP    : explain the power of compounding and interpret long-term wealth accumulation
       - For Budget : Assess the user’s savings rate, classify financial health (Strong / Moderate / Weak), and provide targeted optimization recommendations
       - For Goal   : explain inflation impact, dynamic return choice based on tenure, feasibility against budget surplus, quantify any shortfall, and suggest actionable corrective measures
       - For search : summarize the news/data clearly, extract key numbers, give financial context
       - End every response with one clear, actionable next step

    Constraints and Guidelines:
    - Always use the appropriate tool first. Never guess numbers.
    - Ensure all insights are derived strictly from computed outputs
    - All monetary values must be expressed in Indian Rupees (Rs.)
    - Provide practical, context-aware recommendations rather than generic advice
    - Maintain a balance between analytical precision and user-friendly explanation
    - You have memory of the full conversation — refer back to earlier
      questions or numbers the user has already shared when relevant.
    """
)

print("Finance Agent ready!")
print("Tools:", [t.name for t in finance_tools])

---
## Step 4 — Interactive Chat Loop (with Memory)

This section implements a conversational interface for the Finance AI-Agent with manual memory management. The conversation history is stored and passed to the agent, enabling context-aware responses across multiple user queries.

### Workflow:
- Stores user and assistant messages in conversation memory.
- Sends the complete conversation context to the AI-Agent for reasoning.
- Supports:
  - **Exit:** Terminates the session.
  - **Reset:** Clears stored conversation memory.
  - **Continuous Chat:** Maintains context across interactions.

**Design Principle:**  
Conversation memory enables the AI-Agent to reference previous user inputs and provide personalized financial assistance throughout the session.

In [ ]:
# Manual memory store 
conversation = []

while True:
    user_input = input("\nYou: ")

    if user_input.lower() == "exit":
        break

    if user_input.lower() == "reset":
        conversation = []
        print("Memory cleared!")
        continue

    conversation.append({"role": "user", "content": user_input})

    response = finance_agent.invoke({"messages": conversation})

    assistant_message = response["messages"][-1]
    conversation.append(assistant_message)

    print("Assistant:", assistant_message.content)

---
## Step 5 — Demo Examples

Pre-built example runs covering all 6 tools. Each query builds on the previous one to show how **memory works across turns**.

In [ ]:
demo_conversation = []

def run_demo(query: str):
    """Run a demo query using full chat history and print the response."""
    print(f"\n{'='*55}")
    print(f"User: {query}")
    print(f"{'='*55}")

    demo_conversation.append({"role": "user", "content": query})

    response = finance_agent.invoke({"messages": demo_conversation})

    assistant_message = response["messages"][-1]
    demo_conversation.append(assistant_message)

    print(f"\nAssistant:\n{assistant_message.content}")
    print()

In [ ]:
# Demo 1 — EMI Calculator
run_demo("Calculate EMI for a Rs. 10 lakh loan at 8% interest for 20 years.")

In [ ]:
# Demo 2 — SIP Calculator
run_demo("If I invest Rs. 7000 monthly for 15 years at 10% return, how much will I get?")

In [ ]:
# Demo 3 — Budget Allocator 
run_demo("Create a budget plan for income 90,000.")

In [ ]:
# Demo 4 — General Calculation (PythonRepl)
run_demo("What is 20% of ₹85,000?")

In [ ]:
# Demo 5 
run_demo("How much will I get if I invest ₹3000 per month for 5 years at 8% return")

In [ ]:
# Demo 6 — Integrated Goal Planner
run_demo("I want to save Rs. 20 lakh in 10 years for buying a house. Based on my budget above, can I afford it?")

In [ ]:
# Demo 7 — Financial News Search
run_demo("What is the current RBI repo rate and how does it affect my home loan EMI?")

In [ ]:
# View the full demo conversation history
demo_conversation

### Streamlit UI Code & Link:
https://personal-finance-ai-agent---finai-aj9ysvrme6q4kmomgv8wk2.streamlit.app/